# Datatón CENU — Notebook de arranque (análisis en 15 minutos)

Flujo: **cargar → normalizar → KPIs → sectores → formalidad → género → especialización (LQ/HHI) → espacial → reporte**.

- Si `RUTA_DATOS = None` se usan **datos sintéticos de ensayo** (NO son cifras oficiales).
- El día del evento: suba el archivo entregado por el DANE y ponga su ruta en `RUTA_DATOS`.
- Si el archivo trae **tablas agregadas** (p. ej. municipio × sector × unidades) en vez de microdatos, use `peso="unidades"` en las funciones de indicadores.
- Funciona en Google Colab y en Jupyter local.

In [ ]:
# Configuración (Colab o local)
import sys, os
from pathlib import Path
if "google.colab" in sys.modules and not Path("dataton-dane-2026").exists():
    !git clone -q https://github.com/andresparada-dev/dataton-dane-2026.git
    os.chdir("dataton-dane-2026")
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src")); sys.path.insert(0, str(RAIZ / "scripts"))

import pandas as pd, numpy as np
from cenu import schema, ciiu, indicators as ind, spatial, synthetic, report
pd.set_option("display.max_columns", 50)

In [ ]:
RUTA_DATOS = None          # p. ej. "data/raw/cenu_microdatos.csv"
MAPEO_MANUAL = {}          # p. ej. {"COD_ACT": "ciiu", "P_OCUP": "personal_ocupado"}

if RUTA_DATOS:
    from analisis_rapido import leer
    crudo = leer(RUTA_DATOS, sep=None, hoja=None)
else:
    crudo = synthetic.generar(20_000)   # SINTÉTICO
print(crudo.shape); crudo.head()

In [ ]:
df = schema.normalizar(crudo, mapeo_manual=MAPEO_MANUAL or None)
df = ciiu.enriquecer_ciiu(df)
schema.perfil_calidad(df)

## 1. KPIs de cabecera (primera diapositiva)

In [ ]:
ind.resumen_kpis(df)

## 2. Estructura sectorial y tamaño

In [ ]:
display(ind.conteo_por(df, "gran_sector"))
display(ind.conteo_por(df, ["ciiu_seccion", "ciiu_seccion_nombre"]).head(10))
ind.distribucion_tamano(df)

## 3. Formalidad empresarial
Índice = promedio de % con registro mercantil, % con RUT y % con contabilidad (los disponibles).
Ojo: en los resultados oficiales del CENU el % de RUT se calcula sobre las unidades **con información**.

In [ ]:
display(ind.indice_formalidad(df, "gran_sector"))
ind.indice_formalidad(df, "municipio")

## 4. Brecha de género en la propiedad

In [ ]:
ind.brecha_genero(df, "gran_sector")

## 5. Especialización y diversificación
- **LQ** > 1: el municipio/zona está especializado en esa sección CIIU frente al total.
- **HHI** alto / **Shannon** bajo: economía local concentrada (más vulnerable a choques).

In [ ]:
lq = ind.cociente_localizacion(df, "municipio", "ciiu_seccion", min_unidades=30)
display(lq.round(2).style.background_gradient(cmap="RdBu_r", vmin=0, vmax=2))
ind.hhi_diversificacion(df, "municipio", "ciiu_division")

## 6. Análisis espacial (una ciudad)
Grilla de 500 m → I de Moran (¿se agrupan los negocios?) → Gi* (puntos calientes) → DBSCAN (aglomeraciones).

In [ ]:
ciudad = df["municipio"].value_counts().idxmax()
dc = df[df["municipio"] == ciudad]
g = spatial.grilla(dc, tam_m=500)
coords = g[["latitud", "longitud"]].to_numpy()
print(ciudad, "celdas:", len(g))
print("Moran:", spatial.morans_i(g["valor"].to_numpy(), coords, k=8, permutaciones=199))
g["gi_z"] = spatial.getis_ord_gi_star(g["valor"].to_numpy(), coords, radio_km=1.0)
print("Celdas punto caliente (z>1.96):", (g["gi_z"] > 1.96).sum())
etiquetas = spatial.clusters_dbscan(dc, eps_m=150, min_muestras=15)
spatial.resumen_clusters(dc, etiquetas, "gran_sector").head(10)

## 7. ¿Dónde falta oferta? (brecha de oferta por zona)
Necesita población por zona (proyecciones DANE por municipio o, intra-urbano, CNPV 2018 por manzana/sector).
Aquí se ilustra con la población de las celdas **simulada** — reemplazar por datos reales.

In [ ]:
# Zonas = celdas de 1 km; población ficticia proporcional a viviendas con actividad (SOLO ILUSTRATIVO)
paso = 0.009  # ~1 km en grados
dc = dc.assign(zona=np.floor(dc["latitud"] / paso).astype(int).astype(str) + "_"
                    + np.floor(dc["longitud"] / paso).astype(int).astype(str))
pob = dc.groupby("zona").size().mul(25).rename("poblacion").reset_index()   # <- reemplazar por población real
ind.brecha_oferta(dc, "zona", "ciiu_division", pob, min_esperado=5).head(15)

## 8. Mapa interactivo (opcional, requiere `pip install folium`)

In [ ]:
try:
    import folium
    from folium.plugins import HeatMap
    m = folium.Map(location=[dc["latitud"].mean(), dc["longitud"].mean()], zoom_start=12, tiles="cartodbpositron")
    HeatMap(dc[["latitud", "longitud"]].dropna().astype(float).values.tolist(), radius=8).add_to(m)
    m.save(str(RAIZ / "reports" / "mapa_calor.html")); display(m)
except ImportError:
    print("pip install folium  para el mapa interactivo")

## 9. Reporte automático (HTML autocontenido para mostrar al jurado)

In [ ]:
salida = report.generar_reporte(df, RAIZ / "reports" / "notebook", "Análisis CENU", region="municipio",
                                advertencia=None if RUTA_DATOS else "DATOS SINTÉTICOS — NO OFICIALES")
print(salida / "reporte.html")